# Project 1

In [ ]:
# Install dependencies.
!pip install -q "numpy>=2,<3" tensorflow

## Load and Inspect the Sample Training Data

Load the paired pickle files from `data/` and inspect the first five samples. Each label corresponds to the sample at the same list index.


In [1]:
import pickle
import numpy as np

# Run this notebook from the project1/ directory.
with open("data/CSCE-636-Project-1-Train-n_k_m_P", "rb") as file:
    features = pickle.load(file)

with open("data/CSCE-636-Project-1-Train-mHeights", "rb") as file:
    labels = pickle.load(file)

if len(features) != len(labels):
    raise ValueError(
        f"Sample/label count mismatch: {len(features)} samples, {len(labels)} labels."
    )

first_five_features = features[:5]
first_five_labels = labels[:5]
print(f"Loaded {len(features):,} samples and {len(labels):,} corresponding labels.")

# Print the first five samples and their corresponding labels.
for index, ((n, k, m, P), m_height) in enumerate(
    zip(first_five_features, first_five_labels)
):
    if n != 9 or k not in (4, 5, 6) or not (2 <= m <= n - k):
        raise ValueError(f"Sample at index {index} has invalid n, k, or m.")
    if not isinstance(P, np.ndarray) or P.shape != (k, n - k):
        raise ValueError(f"Sample at index {index}: expected a NumPy array of shape {(k, n - k)}.")

    print("=" * 64)
    print(f"Sample {index + 1} (list index {index})")
    print(f"n = {n}, k = {k}, m = {m}")
    print(f"P.shape = {P.shape}; expected (k, n-k) = {(k, n - k)}")
    print(f"P dtype: {P.dtype}")
    print("P contents:")
    with np.printoptions(threshold=np.inf, floatmode="unique", linewidth=120):
        print(P)
    print(f"True m-height = {m_height!r}")



Loaded 3,833,203 samples and 3,833,203 corresponding labels.
Sample 1 (list index 0)
n = 9, k = 4, m = 4
P.shape = (4, 5); expected (k, n-k) = (4, 5)
P dtype: float64
P contents:
[[  7.  29.  32.  33. -42.]
 [ 24.  38.  -3.  26.  17.]
 [ 28. -35. -20.  32. -29.]
 [-32. -14.   1. -47. -18.]]
True m-height = 415.96346010439976
Sample 2 (list index 1)
n = 9, k = 4, m = 3
P.shape = (4, 5); expected (k, n-k) = (4, 5)
P dtype: float64
P contents:
[[  99.   29.  -98.   81.  -22.]
 [  24.   -3.  -48.   52. -100.]
 [ -34.   38.  -70.   55.   21.]
 [ -78.  -37.   59.  -41.  -32.]]
True m-height = 176.92456578763563
Sample 3 (list index 2)
n = 9, k = 4, m = 2
P.shape = (4, 5); expected (k, n-k) = (4, 5)
P dtype: float64
P contents:
[[-41.   6.  31. -45.  48.]
 [ 10. -19. -39.  40.   1.]
 [-22. -23.   3.  -8.  38.]
 [ 24.  18. -26.  22.  -8.]]
True m-height = 34.156780049509464
Sample 4 (list index 3)
n = 9, k = 4, m = 3
P.shape = (4, 5); expected (k, n-k) = (4, 5)
P dtype: float64
P contents:
[[ 

## Paramter Setup and Configuration

The model predicts $z=\log_2(\hat y)$, and its training target is $\log_2(y)$. The only output conversion is $\hat y=2^z$. A standard bounded ReLU layer restricts $z$ to `[0, 100]`, so this conversion yields a finite value between `1` and `2**100` for finite network outputs. (The upper bound is a broad numerical safeguard, not a value fitted to the test data. Nonfinite network results raise an error before any prediction file is saved.)

In [2]:
import hashlib
import json
import pickle
import numpy as np
from pathlib import Path
import tensorflow as tf

SEED = 636
BATCH_SIZE = 4096
INFERENCE_BATCH_SIZE = 1024
MAX_EPOCHS = 50
MAX_LOG2_HEIGHT = 100.0
ARTIFACT_DIR = Path("artifacts/dnn_v1")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "best_model.keras"
PARAMETER_SETTINGS = [(9, k, m) for k in (4, 5, 6) for m in range(2, 9 - k + 1)]

# Set TensorFlow to be deterministic and reproducible.
tf.keras.utils.set_random_seed(SEED)

# Enable deterministic operations in TensorFlow to ensure reproducibility.
tf.config.experimental.enable_op_determinism()

# Enable GPU memory growth to avoid allocating all GPU memory at once.
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

# Print TensorFlow version and available GPUs.
print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


2026-10-02 15:18:35.521345: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-02 15:18:35.521388: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-02 15:18:35.587111: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-02 15:18:35.824603: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-10-02 15:18:41.078199: W tensorflow/compiler/tf2

TensorFlow: 2.15.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## Input Formatting and Data Splitting

`P` varies in shape, so we place it in the upper-left corner of a zero-padded `6 x 5` array. The original entries and `[n, k, m]` are the only model inputs. Fixed scaling occurs inside standard Keras `Rescaling` layers.

The same matrix can appear under several `m` values. Hence, a row-wise shuffle could place that matrix in both training and testing. Therefore, a seeded hash of `(n, k, P)` assigns each distinct matrix to training, validation, or testing with probabilities **60%, 20%, 20%**. Identical matrix contents always get the same assignment, even if stored as different NumPy objects. Actual row percentages are approximate because whole matrix groups stay together. The hash is used only for partitioning and is never fed into the model.

Nonfinite or below-one training labels have no valid finite log-loss target. We report and exclude such rows from supervised training and local evaluation.

In [3]:
def pack_features(samples):
    matrices = np.zeros((len(samples), 6, 5), dtype=np.float32)
    parameters = np.empty((len(samples), 3), dtype=np.float32)
    for index, (n, k, m, P) in enumerate(samples):
        if n != 9 or k not in (4, 5, 6) or not 2 <= m <= n - k:
            raise ValueError(f"Invalid parameters at row {index}: {(n, k, m)}")
        if not isinstance(P, np.ndarray) or P.shape != (k, n - k):
            raise ValueError(f"Invalid P shape at row {index}")
        matrices[index, :k, :n - k] = P
        parameters[index] = (n, k, m)
    if not np.isfinite(matrices).all():
        raise ValueError("Input matrices must remain finite after float32 conversion")
    return {"matrix": matrices, "parameters": parameters}


def matrix_partitions(samples, seed):
    partitions = np.empty(len(samples), dtype=np.uint8)
    object_cache = {}
    key = int(seed).to_bytes(8, "little")
    for index, (n, k, m, P) in enumerate(samples):
        cache_key = (int(n), int(k), id(P))
        if cache_key not in object_cache:
            payload = bytes((int(n), int(k))) + np.asarray(P, dtype="<f8").tobytes(order="C")
            digest = hashlib.blake2b(payload, digest_size=8, key=key).digest()
            bucket = int.from_bytes(digest, "little") % 10
            object_cache[cache_key] = 0 if bucket < 6 else (1 if bucket < 8 else 2)
        partitions[index] = object_cache[cache_key]
    return partitions


In [4]:
if len(features) != len(labels):
    raise ValueError("Feature and label counts differ")
raw_targets = np.asarray(labels, dtype=np.float64)
if raw_targets.shape != (len(features),):
    raise ValueError("Expected one scalar label per sample")
valid_target = np.isfinite(raw_targets) & (raw_targets >= 1.0)
print(f"Excluded {np.count_nonzero(~valid_target):,} invalid training labels")
if not np.any(valid_target):
    raise ValueError("No valid training labels")

targets_log2 = np.zeros((len(features), 1), dtype=np.float32)
targets_log2[valid_target, 0] = np.log2(raw_targets[valid_target])
if targets_log2[valid_target].max() > MAX_LOG2_HEIGHT:
    raise ValueError("A training target exceeds the model output range")
inputs = pack_features(features)
partition = matrix_partitions(features, SEED)
train_indices = np.flatnonzero(valid_target & (partition == 0))
validation_indices = np.flatnonzero(valid_target & (partition == 1))
test_indices = np.flatnonzero(valid_target & (partition == 2))

setting_ids = np.empty(len(features), dtype=np.int8)
for group_id, (n, k, m) in enumerate(PARAMETER_SETTINGS):
    mask = (inputs["parameters"][:, 1] == k) & (inputs["parameters"][:, 2] == m)
    setting_ids[mask] = group_id

splits = {"train": train_indices, "validation": validation_indices, "test": test_indices}
for name, indices in splits.items():
    if len(indices) == 0:
        raise ValueError(f"The {name} split is empty")
    print(f"{name:10s}: {len(indices):,} rows ({100 * len(indices) / valid_target.sum():.2f}%)")
print("\n(n, k, m)       train       validation       test")
for group_id, setting in enumerate(PARAMETER_SETTINGS):
    counts = [int(np.count_nonzero(setting_ids[indices] == group_id)) for indices in splits.values()]
    if min(counts) == 0:
        raise ValueError(f"Setting {setting} is missing from a split")
    print(f"{str(setting):12s} {counts[0]:10,d} {counts[1]:14,d} {counts[2]:10,d}")
np.savez_compressed(ARTIFACT_DIR / "split_indices.npz", **splits)


Excluded 0 invalid training labels
train     : 2,300,813 rows (60.02%)
validation: 767,229 rows (20.02%)
test      : 765,161 rows (19.96%)

(n, k, m)       train       validation       test
(9, 4, 2)       404,903        134,520    134,475
(9, 4, 3)       403,606        134,082    134,029
(9, 4, 4)       390,478        129,762    129,578
(9, 4, 5)       289,490         96,520     95,727
(9, 5, 2)       204,318         68,891     68,395
(9, 5, 3)       199,481         67,244     66,746
(9, 5, 4)       148,421         50,017     49,392
(9, 6, 2)       151,358         50,173     50,518
(9, 6, 3)       108,758         36,020     36,301


## DNN

The Deep Nerual Network model has three hidden dense layers with 256, 128, and 64 ReLU units. The Softplus output is nonnegative, and the final bounded ReLU enforces the numerical output range. Fixed `P / 100` and parameter scaling are part of the saved model, so training and inference use the same transformation.

We weight training and validation rows inversely by their group's frequency so that equal weights are given to each parameter setting, even though the dataset has unequal group sizes.

In [5]:
def build_model():
    matrix_input = tf.keras.Input(shape=(6, 5), name="matrix")
    parameter_input = tf.keras.Input(shape=(3,), name="parameters")
    matrix_values = tf.keras.layers.Rescaling(1.0 / 100.0)(matrix_input)
    matrix_values = tf.keras.layers.Flatten()(matrix_values)
    parameter_values = tf.keras.layers.Rescaling(1.0 / 10.0)(parameter_input)
    x = tf.keras.layers.Concatenate()([matrix_values, parameter_values])
    for units in (256, 128, 64):
        x = tf.keras.layers.Dense(units, activation="relu")(x)
    x = tf.keras.layers.Dense(1, activation="softplus")(x)
    output = tf.keras.layers.ReLU(max_value=MAX_LOG2_HEIGHT, name="log2_height")(x)
    network = tf.keras.Model(
        inputs={"matrix": matrix_input, "parameters": parameter_input},
        outputs=output, name="mheight_mlp",
    )
    network.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="mean_squared_error",
        # Sample weights already balance the loss; no additional metrics requested.
        weighted_metrics=[],
    )
    return network

model = build_model()
model.summary()


2026-10-02 15:19:05.476520: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 79189 MB memory:  -> device: 0, name: NVIDIA A100-SXM4-80GB, pci bus id: 0000:03:00.0, compute capability: 8.0


Model: "mheight_mlp"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 matrix (InputLayer)         [(None, 6, 5)]               0         []                            
                                                                                                  
 rescaling (Rescaling)       (None, 6, 5)                 0         ['matrix[0][0]']              
                                                                                                  
 parameters (InputLayer)     [(None, 3)]                  0         []                            
                                                                                                  
 flatten (Flatten)           (None, 30)                   0         ['rescaling[0][0]']           
                                                                                        

In [6]:
def setting_weights(indices):
    counts = np.bincount(setting_ids[indices], minlength=len(PARAMETER_SETTINGS))
    if np.any(counts == 0):
        raise ValueError("Each parameter setting must be present in the weighted split")
    return (len(indices) / (len(PARAMETER_SETTINGS) * counts[setting_ids[indices]])).astype(np.float32)


def make_dataset(indices, training=False):
    # Keep the dataset on host memory; only batches move to the training device.
    with tf.device("/CPU:0"):
        dataset = tf.data.Dataset.from_tensor_slices((
            {name: values[indices] for name, values in inputs.items()},
            targets_log2[indices], setting_weights(indices),
        ))
    if training:
        dataset = dataset.shuffle(min(len(indices), 100_000), seed=SEED, reshuffle_each_iteration=True)
    options = tf.data.Options()
    options.experimental_deterministic = True
    options.threading.private_threadpool_size = 2
    return dataset.with_options(options).batch(BATCH_SIZE).prefetch(1)


## Train the DNN

Early stopping and learning-rate reduction use the validation loss.

In [7]:
train_dataset = make_dataset(train_indices, training=True)
validation_dataset = make_dataset(validation_indices)
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(str(MODEL_PATH), monitor="val_loss", save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
    tf.keras.callbacks.TerminateOnNaN(),
]
history = model.fit(
    train_dataset, validation_data=validation_dataset,
    epochs=MAX_EPOCHS, callbacks=callbacks, verbose=2,
    shuffle=False,
)

if not np.isfinite(history.history["val_loss"]).any():
    raise RuntimeError("Training produced no finite validation loss; do not export predictions")
with (ARTIFACT_DIR / "history.json").open("w") as file:
    json.dump({name: [float(value) for value in values] for name, values in history.history.items()}, file, indent=2)
print("Best validation macro log2 MSE:", min(history.history["val_loss"]))
print("Saved model:", MODEL_PATH)


Epoch 1/50


2026-10-02 15:19:10.439505: I external/local_xla/xla/service/service.cc:168] XLA service 0x7fe3ac86d030 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-10-02 15:19:10.439523: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100-SXM4-80GB, Compute Capability 8.0
2026-10-02 15:19:10.447600: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-10-02 15:19:10.499754: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8903
I0000 00:00:1790979550.593894  887507 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


562/562 - 13s - loss: 6.5970 - val_loss: 1.4316 - lr: 0.0010 - 13s/epoch - 23ms/step
Epoch 2/50
562/562 - 9s - loss: 1.2620 - val_loss: 1.1581 - lr: 0.0010 - 9s/epoch - 16ms/step
Epoch 3/50
562/562 - 9s - loss: 1.1237 - val_loss: 1.0895 - lr: 0.0010 - 9s/epoch - 16ms/step
Epoch 4/50
562/562 - 9s - loss: 1.0700 - val_loss: 1.0549 - lr: 0.0010 - 9s/epoch - 16ms/step
Epoch 5/50
562/562 - 9s - loss: 1.0401 - val_loss: 1.0519 - lr: 0.0010 - 9s/epoch - 17ms/step
Epoch 6/50
562/562 - 9s - loss: 1.0186 - val_loss: 0.9992 - lr: 0.0010 - 9s/epoch - 16ms/step
Epoch 7/50
562/562 - 9s - loss: 1.0002 - val_loss: 0.9866 - lr: 0.0010 - 9s/epoch - 16ms/step
Epoch 8/50
562/562 - 9s - loss: 0.9879 - val_loss: 0.9819 - lr: 0.0010 - 9s/epoch - 16ms/step
Epoch 9/50
562/562 - 9s - loss: 0.9746 - val_loss: 0.9793 - lr: 0.0010 - 9s/epoch - 16ms/step
Epoch 10/50
562/562 - 9s - loss: 0.9667 - val_loss: 0.9575 - lr: 0.0010 - 9s/epoch - 16ms/step
Epoch 11/50
562/562 - 9s - loss: 0.9564 - val_loss: 0.9520 - lr: 0.0

## Test the DNN

Inference uses the saved best model with `training=False`.

The only numerical postprocessing is `np.exp2` to convert log₂ predictions to m-heights. Validation checks reject invalid predictions rather than substituting arbitrary values. 

In [8]:
def load_prediction_model(model_path=MODEL_PATH):
    with tf.device("/CPU:0"):
        return tf.keras.models.load_model(str(model_path), compile=False)


def model_log2_batch(network, batch):
    with tf.device("/CPU:0"):
        values = np.asarray(network(batch, training=False)).reshape(-1).astype(np.float64)
    if not np.isfinite(values).all() or np.any(values < 0) or np.any(values > MAX_LOG2_HEIGHT):
        raise ValueError("Model returned an invalid log2 m-height; predictions were not saved")
    return values


def predict_heights(network, samples):
    predictions = np.empty(len(samples), dtype=np.float64)
    for start in range(0, len(samples), INFERENCE_BATCH_SIZE):
        stop = min(start + INFERENCE_BATCH_SIZE, len(samples))
        log_values = model_log2_batch(network, pack_features(samples[start:stop]))
        predictions[start:stop] = np.exp2(log_values)
    if not np.isfinite(predictions).all() or np.any(predictions < 1):
        raise ValueError("Predictions must be finite and at least 1")
    return predictions


def save_prediction_list(path, predictions):
    predictions = np.asarray(predictions, dtype=np.float64)
    if predictions.ndim != 1 or not np.isfinite(predictions).all() or np.any(predictions < 1):
        raise ValueError("Expected a one-dimensional list of finite predictions >= 1")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    with temporary.open("wb") as file:
        pickle.dump(predictions.tolist(), file, protocol=pickle.HIGHEST_PROTOCOL)
    temporary.replace(path)


In [9]:
inference_model = load_prediction_model()

def report_costs(name, indices):
    predicted_log2 = np.empty(len(indices), dtype=np.float64)
    for start in range(0, len(indices), INFERENCE_BATCH_SIZE):
        selected = indices[start:start + INFERENCE_BATCH_SIZE]
        batch = {key: values[selected] for key, values in inputs.items()}
        predicted_log2[start:start + len(selected)] = model_log2_batch(inference_model, batch)
    # Use the original float64 labels for reporting the specified cost.
    errors = (np.log2(raw_targets[indices]) - predicted_log2) ** 2
    rows = []
    print(f"\n{name}: per-setting squared log2 error")
    for group_id, setting in enumerate(PARAMETER_SETTINGS):
        mask = setting_ids[indices] == group_id
        cost = float(errors[mask].mean())
        rows.append({"n": setting[0], "k": setting[1], "m": setting[2],
                     "count": int(mask.sum()), "sigma": cost})
        print(f"{setting}: count={mask.sum():,}, sigma={cost:.6f}")
    macro = float(np.mean([row["sigma"] for row in rows]))
    print(f"Equal-setting mean cost: {macro:.6f}")
    print(f"Row-weighted mean cost: {errors.mean():.6f}")
    return {"per_setting": rows, "macro_sigma": macro, "row_mean_sigma": float(errors.mean())}

cost_report = {
    "validation": report_costs("Validation", validation_indices),
    "local_test": report_costs("Held-out test", test_indices),
}
with (ARTIFACT_DIR / "costs.json").open("w") as file:
    json.dump(cost_report, file, indent=2)



Validation: per-setting squared log2 error
(9, 4, 2): count=134,520, sigma=0.093246
(9, 4, 3): count=134,082, sigma=0.141924
(9, 4, 4): count=129,762, sigma=0.418401
(9, 4, 5): count=96,520, sigma=2.206490
(9, 5, 2): count=68,891, sigma=0.110310
(9, 5, 3): count=67,244, sigma=0.392302
(9, 5, 4): count=50,017, sigma=2.245658
(9, 6, 2): count=50,173, sigma=0.283523
(9, 6, 3): count=36,020, sigma=1.885558
Equal-setting mean cost: 0.864157
Row-weighted mean cost: 0.687252

Held-out test: per-setting squared log2 error
(9, 4, 2): count=134,475, sigma=0.092570
(9, 4, 3): count=134,029, sigma=0.142509
(9, 4, 4): count=129,578, sigma=0.420238
(9, 4, 5): count=95,727, sigma=2.231532
(9, 5, 2): count=68,395, sigma=0.109915
(9, 5, 3): count=66,746, sigma=0.398978
(9, 5, 4): count=49,392, sigma=2.267675
(9, 6, 2): count=50,518, sigma=0.281212
(9, 6, 3): count=36,301, sigma=1.862818
Equal-setting mean cost: 0.867494
Row-weighted mean cost: 0.689530
